In [1]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
import os
import requests
from dotenv import load_dotenv
from typing import Literal
from urllib.parse import quote_plus
from pydantic import BaseModel, Field
import feedparser

In [9]:
import os
import json
import statistics
import requests
import feedparser
from datetime import date, timedelta
from typing import Literal
from urllib.parse import quote_plus
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables (ensure GEMINI_API_KEY is set in your .env file)
load_dotenv()

API = 'https://api.frankfurter.dev/v2'

# Initialize Gemini via LangChain
llm = ChatGoogleGenerativeAI(
    model=os.getenv('MODEL_NAME', 'gemini-3.1-flash-lite'), temperature=0
)


# Define a Pydantic schema to force strict structured output from the LLM
class NewsView(BaseModel):
  sentiment: Literal['positive', 'negative', 'neutral']
  recommendation: Literal['buy', 'hold', 'sell'] = Field(description=('buy if condition favor buying, selling if condition favor selling,'
                                              'and holding if condition favor holding'))
  reason: str  # One-sentence justification


news_judge = llm.with_structured_output(NewsView)


def get_rates(base='USD', targets=None):
  if targets is None:
    targets = ['EUR', 'GBP', 'CAD', 'npr', 'INR', 'AUD']

  url = f'{API}/rates'
  params = {'base': base, 'quotes': ','.join(targets)}
  response = requests.get(url, params=params, timeout=10)
  response.raise_for_status()

  # The API returns a list directly
  rows = response.json()
  rates_list = []
  date_str = None

  for row in rows:
    date_str = row.get('date')
    rates_list.append({'quote': row['quote'], 'rate': row['rate']})

  return {'date': date_str, 'rates': rates_list}


def get_headlines(currency_code, n=3):
  query = quote_plus(f'{currency_code} currency exchange rate when:3d')
  rss_url = f'https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en'
  feed = feedparser.parse(rss_url)
  return [entry.title for entry in feed.entries[:n]]


def judge_news(currency_code, base_currency, titles):
  if not titles:
    return None
  prompt = (
      f'Headlines about {currency_code} from the last few days:\n'
      + '\n'.join(titles)
      + f'\n\nIs the news positive, negative or neutral for the value of {currency_code} against {base_currency}? '
      'Judge only from these headlines. tell me wether i should buy the currency or not.'
  )
  return news_judge.invoke(prompt)


def main():
  base_currency = 'USD'
  print(
      f'--- Live Exchange Rates & AI Sentiment (Base: {base_currency}) ---\n'
  )

  try:
    data = get_rates(base=base_currency)
    print(f'Date: {data.get('date')}\n')

    for item in data.get('rates', []):
      currency = item['quote']
      rate = item['rate']
      print(f'1 {base_currency} = {rate:.4f} {currency}')

      # Fetch recent news headlines for the currency
      headlines = get_headlines(currency, n=3)

      if headlines:
        print('Recent Headlines:')
        for title in headlines:
          print(f'      • {title}')

        # Ask Gemini to judge the sentiment based on the headlines
        view = judge_news(currency, base_currency, headlines)
        if view:
          print(
              f'AI Sentiment: [{view.sentiment.upper()}] —'
              f' {view.reason}'
          )
      else:
        print('No recent headlines found.')

      print('-' * 60)

  except Exception as e:
    print(f'Error: {e}')


if __name__ == '__main__':
  main()

--- Live Exchange Rates & AI Sentiment (Base: USD) ---

Date: 2026-11-15

1 USD = 1.4694 AUD
Recent Headlines:
      • Official exchange rates on selected date | Bank of Russia - Центральный Банк России
      • Australian Dollar to U.S. Dollar - AUD/USD Exchange Rate Chart - FOREX.com
      • 11 Currencies That Move Most Closely With Oil Prices - EBC Financial Group
AI Sentiment: [NEUTRAL] — The provided headlines are informational and do not contain specific economic data, policy shifts, or market sentiment indicators that would suggest a directional move for the AUD/USD pair.
------------------------------------------------------------
1 USD = 1.4238 CAD
Recent Headlines:
      • Official exchange rates on selected date | Bank of Russia - Центральный Банк России
      • Canadian Dollar’s Weakening Ties to Oil Prices Amplifies Bets Against It - EnergyNow
      • USD exchange rate today, November 15th: The US dollar stabilizes after a surge; what should you keep in mind when buying USD

KeyboardInterrupt: 